# Introduction to the Malthusian Population Growth Model

## What is the Malthusian Model?

In 1798, the English clergyman and economist Thomas Robert Malthus published *An Essay on the Principle of Population*, arguing that human populations, when left unchecked, grow **exponentially** — doubling at a roughly constant rate — while food production can only grow **arithmetically**. This tension between exponential growth and limited resources is the core of the Malthusian thesis.

The mathematical model that bears his name is elegantly simple. If $P(t)$ is the population at time $t$, then the rate of change of the population is proportional to the population itself:

$$\frac{dP}{dt} = r \, P(t)$$

where $r$ is the **intrinsic growth rate** (birth rate minus death rate). Solving this first-order ODE gives:

$$P(t) = P_0 \, e^{r(t - t_0)}$$

Here $P_0 = P(t_0)$ is the population at reference time $t_0$. This is the **exponential growth law**.

For $r > 0$, population grows without bound. In practice, populations slow down due to declining fertility, resource constraints, or demographic transition. But for countries in early stages of development, or over shorter time windows, the Malthusian model can be a surprisingly good fit — and understanding *where* it holds and *where* it breaks down is the central question of this project.

## Why Does Exponential Growth Feel Surprising?

Exponential growth is famously counter-intuitive. A classic illustration is the **wheat and chessboard problem**: place 1 grain of wheat on the first square, 2 on the second, 4 on the third, and so on doubling each time. By the 64th square, you need more wheat than has ever been grown in all of history.

For populations, a useful rule of thumb is the **rule of 70**: a population growing at $r\%$ per year doubles in approximately $70 / r$ years. A 2% annual growth rate doubles a population in about 35 years; at 3%, doubling takes only 23 years.

Small differences in $r$ compound into enormous differences over decades — which is why demographic projections matter so much for policy planning.

## The Malthusian Model vs. Alternatives

The Malthusian model is the simplest member of a family of population models:

| Model | Equation | Key Assumption | Limitation |
|---|---|---|---|
| **Malthusian (Exponential)** | $P(t) = P_0 e^{rt}$ | Growth rate $r$ is constant | Predicts infinite growth |
| **Logistic (Verhulst)** | $\frac{dP}{dt} = rP\left(1 - \frac{P}{K}\right)$ | Growth slows as $P \to K$ | Assumes a fixed carrying capacity $K$ |
| **Demographic Transition** | Multi-stage qualitative model | Birth/death rates shift with development | Hard to parameterise precisely |

In your capstone project you will use the Malthusian model as the baseline. Some of the analytical extensions ask you to compare it against alternatives like the logistic model, so keep this table in mind.

## Key Parameters and Their Meaning

The Malthusian model has just two free parameters:

* **$P_0$** — the population at the reference year $t_0$. This anchors the curve vertically.
* **$r$** — the intrinsic growth rate per year. This controls how steeply the curve rises (or falls). $r$ can be negative for shrinking populations.

A critical observation: taking the natural logarithm of both sides,

$$\ln P(t) = \ln P_0 + r \, (t - t_0)$$

This is a **straight line** in $\ln P$ versus $t$ space. The intercept is $\ln P_0$ and the slope is $r$. This means: if you plot population on a **log scale** and the points fall on a straight line, the country's growth is Malthusian. Curvature on the log scale means the model is struggling.

> **Before fitting any model, always plot your data — on both a linear and a log scale.** This simple step tells you immediately whether the Malthusian assumption is even plausible.

## How Do We Fit a Model to Data?

"Fitting a model" means finding the parameter values — here $(P_0, r)$ — that make the model's predictions match the observed data as closely as possible. But what does "as closely as possible" mean, exactly? This section walks through the core idea step by step.

### Step 1: Define a Residual

Suppose we have observed population values $P_1, P_2, \ldots, P_n$ at years $t_1, t_2, \ldots, t_n$. For a given choice of $(P_0, r)$, the model predicts $\hat{P}_i = P_0 e^{r(t_i - t_0)}$. The **residual** at point $i$ is the gap between observation and prediction:

$$e_i = P_i - \hat{P}_i$$

A positive residual means the model under-predicted; negative means it over-predicted. A perfect model would have all residuals equal to zero.

### Step 2: Define a Loss Function

We want *all* residuals to be small simultaneously. The standard approach is to minimise the **sum of squared residuals (SSR)**:

$$\text{SSR}(P_0, r) = \sum_{i=1}^{n} \left(P_i - P_0 e^{r(t_i - t_0)}\right)^2$$

Why square the residuals?
- Squaring makes all terms positive, so over- and under-predictions do not cancel out.
- Squaring penalises large errors more heavily than small ones — a residual of 10 contributes 100 to the sum, while a residual of 1 contributes only 1.

This is the **least squares** criterion, the most widely used fitting objective in science.

### Step 3: Minimise the Loss

We need the values of $(P_0, r)$ that minimise SSR. For a straight line, a closed-form solution exists (you may have seen these formulas in a statistics course). For a nonlinear function like $P_0 e^{r t}$, there is no simple closed form. Instead we use an **iterative numerical optimiser**.

The optimiser starts from an initial guess and repeatedly adjusts the parameters in the direction that most reduces SSR, until improvements become negligibly small. This is conceptually similar to gradient descent in machine learning.

In Python, `scipy.optimize.curve_fit` does this for us. You provide the model function, the observed data, and an initial guess — and it returns the best-fit parameters along with a **covariance matrix** from which you can read off uncertainty estimates.

### Step 4: Assess the Fit — Goodness-of-Fit Metrics

Minimising SSR gives you the *best possible* Malthusian fit for that dataset. But "best possible" does not mean *good*. You still need to check whether the resulting fit is actually useful. There is no single perfect metric — each captures a different aspect of fit quality, and understanding the trade-offs between them is an important part of doing good data analysis.

We will use three metrics: $R^2$, RMSE, and MAPE. Each is defined below.

---

#### $R^2$ — Coefficient of Determination

$$R^2 = 1 - \frac{\text{SSR}}{\text{SST}}, \quad \text{where} \quad \text{SST} = \sum_{i=1}^n (P_i - \bar{P})^2$$

$\text{SST}$ (total sum of squares) measures how much the data varies around its own mean. The ratio $\text{SSR}/\text{SST}$ is therefore the *fraction of variance left unexplained* by the model. $R^2$ is one minus that fraction — the *fraction of variance explained*.

- $R^2 = 1$: perfect fit — the model explains all variation in the data.
- $R^2 = 0$: the model explains nothing; it is no better than predicting $\bar{P}$ every time.
- $R^2 < 0$: the model is *worse* than the mean predictor — a sign something has gone seriously wrong.

**When $R^2$ can mislead.** A very high $R^2$ during the training period does not mean the model will project forward correctly. Population totals grow over time, so SST tends to be large, and even a curve that is structurally wrong can still achieve a high $R^2$ if it roughly tracks the general upward trend. Always pair $R^2$ with a visual inspection of the fit.

---

#### RMSE — Root Mean Squared Error

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n \left(P_i - \hat{P}_i\right)^2}$$

RMSE is the square root of the average squared residual. Because we undo the squaring at the end, RMSE is expressed in the **same units as the original data** — here, millions of people. An RMSE of 5.0 means the model's predictions are off by roughly 5 million people on average (with larger errors penalised more heavily than smaller ones, due to the squaring).

**When RMSE is useful.** RMSE is easy to interpret in absolute terms and is the right metric when the cost of a large error is disproportionately high (e.g. for policy planning, being off by 50 million is much worse than being off by 5 million twice). 

**When RMSE is misleading.** A country with 1 billion people and an RMSE of 20 million may be fitting much better *proportionally* than a country with 5 million people and an RMSE of 1 million. Raw RMSE values are not directly comparable across countries of very different sizes.

---

#### MAPE — Mean Absolute Percentage Error

$$\text{MAPE} = \frac{1}{n} \sum_{i=1}^n \left| \frac{P_i - \hat{P}_i}{P_i} \right| \times 100\%$$

MAPE expresses each residual as a **percentage of the observed value**, then averages those percentages. A MAPE of 3% means the model's predictions are off by 3% of the true population on average — regardless of whether that country has 5 million or 500 million people.

**When MAPE is useful.** Its scale-independence makes it ideal for comparing fit quality across countries of very different sizes, which is exactly what we need here. A MAPE of 2% is a 2% fit whether the country is tiny or enormous.

**When MAPE is misleading.** If any observed value $P_i$ is close to zero, the percentage explodes — a residual of 0.1 million on an observed value of 0.2 million gives 50%, which distorts the average. For the population data in this project (all values are in the millions), this is not a concern, but it is worth bearing in mind for other applications.

---

#### Choosing the Right Metric

| Metric | Units | Scale-independent? | Best used when… |
|---|---|---|---|
| $R^2$ | none | yes | Comparing fit quality for the *same* country across different models |
| RMSE | millions of people | **no** | Absolute accuracy matters and countries are similar in size |
| MAPE | % | yes | Comparing fit quality *across* countries of different sizes |

In your capstone report, you should **use at least two metrics** and explicitly discuss what each one tells you — and where it might give a misleading impression.

> **Think about it:** Can you construct a scenario where $R^2$ is high but MAPE is also high? What would the residual plot look like in that case?

### Illustrated Example: Least-Squares Linear Fit

Before we fit the Malthusian model — which is nonlinear — it helps to see the same ideas applied to the simplest possible case: **fitting a straight line** $\hat{y} = a + bx$ to a handful of points.

The cell below generates a small synthetic dataset and fits a line to it using `numpy.polyfit`. It then produces two panels:

- **Left — Fit and residuals:** The fitted line is drawn through the data. Each **residual** is shown as a vertical red segment connecting an observed point to the line. The length of the segment is $|e_i| = |y_i - \hat{y}_i|$. Least squares minimises the sum of the *squared* lengths of these segments.
- **Right — Residual plot:** The residuals $e_i$ plotted against $x$. A good fit leaves no pattern here — the residuals should scatter randomly around zero. Any systematic curve or trend means the model is missing structure in the data.

Study the output carefully. The same logic — and the same two diagnostic plots — will apply when you fit the Malthusian exponential model to population data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Synthetic data ────────────────────────────────────────────────────────────
rng  = np.random.default_rng(42)
x    = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
y    = 2.5 * x + 4.0 + rng.normal(scale=2.5, size=len(x))  # true line + noise

# ── Least-squares fit (degree-1 polynomial = straight line) ──────────────────
coeffs      = np.polyfit(x, y, deg=1)      # returns [slope, intercept]
slope, intercept = coeffs
x_line      = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
y_line      = slope * x_line + intercept

# ── Residuals ─────────────────────────────────────────────────────────────────
y_hat       = slope * x + intercept        # predicted values at each data point
residuals   = y - y_hat                    # e_i = observed − predicted

# ── Goodness of fit ───────────────────────────────────────────────────────────
ss_res = np.sum(residuals ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2     = 1 - ss_res / ss_tot

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: data, fitted line, and residual segments
ax = axes[0]
ax.plot(x_line, y_line, color='steelblue', lw=2,
        label=f'Fit: $\\hat{{y}}$ = {slope:.2f}$x$ + {intercept:.2f}')
ax.scatter(x, y, color='black', zorder=4, s=50, label='Observed $y_i$')
for xi, yi, yhi in zip(x, y, y_hat):          # draw each residual segment
    ax.plot([xi, xi], [yi, yhi],
            color='crimson', lw=1.5, linestyle='--')
# Label just one residual to keep the plot readable
mid = len(x) // 2
ax.annotate('', xy=(x[mid], y_hat[mid]), xytext=(x[mid], y[mid]),
            arrowprops=dict(arrowstyle='<->', color='crimson', lw=1.5))
ax.text(x[mid] + 0.15, (y[mid] + y_hat[mid]) / 2,
        '$e_i = y_i - \\hat{y}_i$', color='crimson', fontsize=10, va='center')
ax.set_title(f'Least-Squares Fit  ($R^2$ = {r2:.3f})', fontsize=12)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Right: residual plot
ax = axes[1]
ax.axhline(0, color='steelblue', lw=1.5, linestyle='--', label='Zero line (perfect fit)')
ax.scatter(x, residuals, color='crimson', zorder=4, s=50, label='Residual $e_i$')
for xi, ei in zip(x, residuals):               # drop lines to zero
    ax.plot([xi, xi], [0, ei], color='crimson', lw=1, linestyle=':')
ax.set_title('Residual Plot  (should look random around 0)', fontsize=12)
ax.set_xlabel('$x$'); ax.set_ylabel('Residual  $e_i = y_i - \\hat{y}_i$')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Fitted line :  ŷ = {slope:.3f} · x + {intercept:.3f}")
print(f"True line   :  ŷ = 2.500 · x + 4.000")
print(f"SSR = {ss_res:.2f}    R² = {r2:.4f}")

#### Things to notice

- The fitted slope and intercept are close to the true values (2.5 and 4.0) but not identical — the noise in the data shifts them slightly. This is always true in practice: you recover an *estimate* of the underlying parameters, not the exact truth.
- The residual plot looks scattered with no obvious pattern — this is what you want. If you saw a U-shaped curve in the residual plot, it would mean a straight line is the wrong model and something nonlinear is needed.
- Some residuals are positive, some negative. Least squares does not care about the sign — it only minimises their *squared* sum, so both directions are penalised equally.

> **Try it:** Change `scale=2.5` to `scale=8.0` in the noise term and rerun. How does a noisier dataset affect the fitted line, the residuals, and $R^2$? What if you set `scale=0.1`?

## Python Implementation

### Setup

Run the cell below to import the required libraries and define the time constants for this analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ==========================================
# Time constants
# ==========================================
REFERENCE_YEAR = 1955   # t = 0 anchor — P0 is the population in this year
FIT_END_YEAR   = 2000   # fit on 1955–2000 ...
DATA_END_YEAR  = 2020   # ... and see how it projects to 2020

PALETTE = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']  # one colour per country

### The Dataset

Below are population figures (in millions) at 5-year intervals for **four anonymised countries**, labelled Alpha, Beta, Gamma, and Delta. The data spans 1955 to 2020.

The four countries were chosen to represent a range of demographic histories — some follow the Malthusian model closely, others deviate markedly. Part of your job is to work out which is which, and to think about *why*.

> **Before running any fitting code:** Look at the raw numbers. Can you already form a hypothesis about which country might be most Malthusian? What features in the table suggest exponential growth versus slowing or levelling-off growth?

In [ ]:
YEARS = np.array([1955, 1960, 1965, 1970, 1975, 1980, 1985, 1990,
                  1995, 2000, 2005, 2010, 2015, 2020])

POPULATION_DATA = {
    "Alpha":  np.array([ 38.1,  45.1,  53.7,  64.0,  76.3,  72.9,  86.5, 103.0,
                        120.9, 140.4, 162.5, 190.9, 224.3, 206.2]),
    "Beta":   np.array([ 89.0,  93.4,  98.2, 103.7, 110.9, 116.8, 120.8, 123.5,
                        125.5, 126.9, 127.8, 128.1, 127.1, 125.7]),
    "Gamma":  np.array([ 62.6,  72.8,  84.7,  96.4, 108.1, 121.7, 136.2, 150.7,
                        163.7, 175.0, 186.8, 198.6, 207.8, 212.6]),
    "Delta":  np.array([  3.3,   3.8,   4.4,   5.1,   5.9,   6.9,   8.0,   9.3,
                          10.9,  12.8,  15.2,  18.2,  21.9,  24.2]),
}

### Visualising the Raw Data

Always look at your data before fitting anything. The cell below produces two plots side by side:

- **Left (linear scale):** Shows the raw trajectories. Exponential curves look like hockey sticks here.
- **Right (log scale):** If growth is truly exponential, the points should fall on a **straight line**. Any bend or curve on the log scale signals that the Malthusian model is not a perfect fit.

> **Task:** Before running this cell, write down your prediction: which country do you expect to look most linear on the log scale? After running it, compare your prediction to the plot. Were you right? If not, what did you miss?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for idx, (country, pop) in enumerate(POPULATION_DATA.items()):
    c = PALETTE[idx]
    axes[0].plot(YEARS, pop,     marker='o', markersize=4, color=c, label=country)
    axes[1].semilogy(YEARS, pop, marker='o', markersize=4, color=c, label=country)

axes[0].set_title("Population Over Time — Linear Scale", fontsize=12)
axes[1].set_title("Population Over Time — Log Scale", fontsize=12)
for ax in axes:
    ax.set_xlabel("Year")
    ax.set_ylabel("Population (millions)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Reflection Questions

Take a moment before moving on to fitting:

1. On the log scale, which countries appear most linear over the full 1955–2020 range? Which ones curve?
2. Are there countries that look linear *early* in the period but then bend? At roughly what year does the bend appear?
3. One country's trajectory appears to flatten or even turn downward. What would that look like on the log scale versus the linear scale?
4. Does the absolute size of a country's population (large vs. small) affect how well the Malthusian model fits? Why or why not?

You do not need to write answers here, but these are exactly the kinds of questions your final report should address.

### The Model Function

We define the Malthusian growth curve as a Python function. Time is measured **relative to `REFERENCE_YEAR`**, so $t = 0$ corresponds to 1955 and the parameter `P0` is directly interpretable as the 1955 population.

In [ ]:
def malthusian(t, P0, r):
    """
    Malthusian (exponential) growth model.

    Parameters
    ----------
    t  : array-like  years
    P0 : float       population at REFERENCE_YEAR (millions)
    r  : float       intrinsic growth rate per year

    Returns
    -------
    Predicted population (millions).
    """
    return P0 * np.exp(r * (np.asarray(t) - REFERENCE_YEAR))


# Sanity check — with r = 0.02, a population should roughly double in 35 years
t_check = np.array([REFERENCE_YEAR, REFERENCE_YEAR + 35])
P_check = malthusian(t_check, P0=100.0, r=0.02)
print(f"Sanity check (r=0.02, P0=100):")
print(f"  {t_check[0]}: {P_check[0]:.1f}M  →  {t_check[1]}: {P_check[1]:.1f}M")
print(f"  Ratio: {P_check[1]/P_check[0]:.3f}  (expected ≈ {np.exp(0.02*35):.3f})")

#### A Note on `curve_fit` and Initial Guesses

`scipy.optimize.curve_fit(f, xdata, ydata, p0)` finds the parameters that minimise the sum of squared residuals. Internally it uses the **Levenberg–Marquardt algorithm**, which iteratively adjusts the parameters until it cannot reduce SSR any further.

It returns:
- `popt` — the best-fit parameters `[P0, r]`
- `pcov` — the covariance matrix; `np.sqrt(np.diag(pcov))` gives the standard error on each parameter

**The initial guess `p0` matters.** If your starting point is far from the true minimum, the optimiser may converge to a wrong or nonsensical solution. A natural choice for `P0` is the first observed population value; for `r`, a small positive number like `0.02` is reasonable for most countries. 

> **Try this:** After running the fit, change the initial guess to something very different — say `p0 = [1.0, 0.5]` — and rerun. Does the result change? If it does, that is a warning sign that the optimisation landscape has multiple local minima.

### Fitting the Model

The cell below fits the Malthusian model to the **training data** (1955–2000) for each country. The **projection period** (2000–2020) is held back — we will use it to test how well the model generalises beyond the data it was fitted on.

For each country the code computes:
- The fitted parameters $P_0$ and $r$, together with their standard errors from `pcov`
- **$R^2$** — fraction of variance explained (dimensionless, 0 to 1)
- **RMSE** — root mean squared error in millions of people
- **MAPE** — mean absolute percentage error (scale-free, in %)

All three metrics are defined and discussed in the *Step 4* section above. Refer back to that discussion when interpreting the table.

In [ ]:
train_mask  = YEARS <= FIT_END_YEAR
years_train = YEARS[train_mask]

results = {}

for country, pop in POPULATION_DATA.items():
    pop_train = pop[train_mask]

    # Fit — initial guess: observed 1955 population, 2% annual growth
    p0_guess = [pop_train[0], 0.02]
    popt, pcov = curve_fit(malthusian, years_train, pop_train,
                           p0=p0_guess, maxfev=5000)
    P0_fit, r_fit = popt
    P0_std, r_std = np.sqrt(np.diag(pcov))

    # Predicted values on training data
    pop_pred = malthusian(years_train, *popt)

    # Goodness-of-fit statistics
    ss_res = np.sum((pop_train - pop_pred) ** 2)
    ss_tot = np.sum((pop_train - pop_train.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot
    rmse   = np.sqrt(np.mean((pop_train - pop_pred) ** 2))
    mape   = np.mean(np.abs((pop_train - pop_pred) / pop_train)) * 100

    results[country] = dict(P0=P0_fit, P0_std=P0_std,
                            r=r_fit,   r_std=r_std,
                            R2=r2, RMSE=rmse, MAPE=mape)

# Summary table
print(f"{'Country':<8} {'P₀ (M)':>9} {'±':>6} {'r (yr⁻¹)':>10} {'±':>7} {'R²':>7} {'RMSE':>7} {'MAPE%':>7}")
print("-" * 65)
for country, m in results.items():
    print(f"{country:<8} {m['P0']:>9.1f} {m['P0_std']:>6.1f} "
          f"{m['r']:>10.4f} {m['r_std']:>7.4f} "
          f"{m['R2']:>7.4f} {m['RMSE']:>7.2f} {m['MAPE']:>7.2f}")

### Reading the Table

Now that you have the three metrics defined, here is how to read each column:

**Parameter columns ($P_0$, $r$, and their `±` standard errors)**

The standard errors tell you how precisely the data constrains each parameter. A small standard error on $r$ means the 1955–2000 data strongly implies a particular growth rate. A large standard error means the fit is ambiguous — different values of $r$ produce almost equally good SSR, so you should be cautious about interpreting the estimated $r$ as a reliable biological or demographic quantity.

**$R^2$**

Compare $R^2$ across countries. A value above 0.99 suggests the Malthusian model is an excellent description of that country's trajectory over the training period. A value below 0.95 should prompt you to look carefully at the fitted curve — something systematic may be going on.

Be careful, though: a country whose population barely changes over 45 years (like Beta) will have a very small SST. Any smooth curve that stays roughly flat will achieve a high $R^2$ — not because exponential growth is the right model, but because there is almost no variance to explain. High $R^2$ alone is not evidence that the *Malthusian* model is appropriate.

**RMSE**

Notice that RMSE is not directly comparable across countries of different sizes. A country with 1 billion people and RMSE = 8 million is fitting proportionally much better than a small country with RMSE = 2 million. Use RMSE to compare *different models on the same country*, not to rank countries against each other.

**MAPE**

MAPE is the right tool for cross-country comparison. A MAPE of 1% is a very tight fit; a MAPE above 5–10% suggests the model is missing something meaningful. Because MAPE is expressed as a percentage of the observed value, it is not inflated by large absolute populations.

> **Question:** Look at the MAPE and $R^2$ columns together. Do they tell the same story for every country, or are there cases where one metric says "good fit" while the other says "poor fit"? If so, which metric do you trust more in that case, and why?

### Visualising the Fits

Numbers alone can be misleading. The cell below plots each country's observed population alongside the fitted Malthusian curve. The **solid line** shows the training window; the **dashed line** shows the projection beyond 2000. The dotted vertical line marks the boundary.

In [ ]:
years_smooth = np.linspace(REFERENCE_YEAR, DATA_END_YEAR, 400)
train_idx    = years_smooth <= FIT_END_YEAR

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, (country, pop) in enumerate(POPULATION_DATA.items()):
    ax = axes[idx]
    c  = PALETTE[idx]
    m  = results[country]

    ax.scatter(YEARS, pop, color=c, s=30, zorder=3, label="Observed")

    y_curve = malthusian(years_smooth, m['P0'], m['r'])
    ax.plot(years_smooth[train_idx],  y_curve[train_idx],  color=c, lw=2,
            label=f"Fit  r={m['r']:.3f} yr⁻¹")
    ax.plot(years_smooth[~train_idx], y_curve[~train_idx], color=c, lw=2,
            linestyle='--', label="Projection")

    ax.axvline(FIT_END_YEAR, color='grey', lw=0.8, linestyle=':')
    ax.set_title(f"{country}  ($R^2$ = {m['R2']:.3f})", fontsize=11)
    ax.set_xlabel("Year")
    ax.set_ylabel("Population (millions)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Malthusian Model: Training Fit (solid) and Projection (dashed)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### What to Look for in These Plots

The plots can reveal things that the numbers alone conceal:

- Is the fitted curve **systematically above or below** the data points during some periods? Systematic patterns in residuals indicate that the model is missing something structural about the population's behaviour.
- How does the **dashed projection** compare to the observed 2000–2020 data? Does it diverge immediately, or track the data for a while before going wrong?
- For a country with a high $R^2$, does the projection still look reasonable? For a country with a lower $R^2$, is the projection already off within the training window?

> **Tip for your report:** A figure like this is worth a thousand words, but only if you *discuss* what it shows. Do not just include plots — interpret them. Where does the model succeed, where does it fail, and what might explain the difference?

## Where to Go from Here

This notebook has set up the scaffolding. Your capstone project requires you to go substantially further. Here are some concrete directions and questions to orient your thinking.

### Expanding the Analysis

- The project requires **at least 10 countries**. Collect your own data from the [UN World Population Prospects](https://population.un.org/wpp/) database or the World Bank API. Think carefully about *which* countries to include — a thoughtful selection will span different regions, sizes, and developmental stages, and that diversity will make your analysis more interesting.
- The four countries here are anonymised to avoid premature conclusions. In your capstone you will work with real, identified datasets and will be expected to bring in historical and geographical context.

### Questions Worth Investigating

- **Does fit quality correlate with income level or region?** High-income countries have largely completed the demographic transition; low-income countries may still be in the exponential phase. Does your $R^2$ data support this pattern?
- **How sensitive is the fitted $r$ to the training window?** Try fitting on 1955–1975 only and compare $r$ to the value from the full 1955–2000 window. If they differ substantially, what does that tell you about whether the Malthusian model is appropriate?
- **When does the projection break down?** For countries with poor out-of-sample performance, try to identify the *year* at which the curve diverges from observed data. Does this coincide with any known historical events — a fertility policy, a conflict, an economic transition?
- **What drives a negative $r$?** The Malthusian model cannot distinguish between falling birth rates, rising death rates, and emigration — they all look the same to the model. Discuss this as a limitation in your report.

### On Writing Up Your Results

- **Justify every methodological choice.** Why those 10+ countries? Why MAPE rather than (or alongside) RMSE? Why a 1955–2000 training window? Good scientific writing explains the reasoning behind decisions, not just the decisions themselves.
- **Show residuals, not just fits.** An $R^2$ value is a single number; a residual plot shows you *where* and *how* the model fails. Consider adding residual plots for each country.
- **Discuss uncertainty.** The standard errors on $r$ from `curve_fit` are a starting point. Countries with similar $R^2$ values may have very different parameter uncertainties — why might that be?

> **Final thought:** A model that fits the training data perfectly but cannot project forward is not useful. A model that projects forward well but fits poorly in-sample might just be lucky. The most informative analysis considers both, and tries to understand the *reasons* behind the differences — not just report the numbers.

## Summary

In this notebook you have:

- Derived the Malthusian growth law and seen why a **log-scale plot** is the natural first diagnostic for exponential growth.
- Walked through the logic of **least-squares fitting** step by step — what a residual is, why we square them, and how an iterative optimiser finds the best parameters.
- Loaded population data for four anonymised countries and fitted the Malthusian model using `scipy.optimize.curve_fit`.
- Evaluated the fit using $R^2$, RMSE, and MAPE, and inspected the results visually.

The analysis here is a starting point, not a finished product. Your capstone project requires you to substantially expand the country sample, deepen the analysis, implement two analytical extensions, and write a structured report that interprets — not just reports — your findings.

## Recommended Reading & Journal Club

### Foundational Texts

**1. Malthus, T. R. (1798)**
*An Essay on the Principle of Population.*
J. Johnson, London.
→ The original argument. Widely available as a free ebook. The first two chapters provide essential historical context.

**2. Pearl, R. & Reed, L. J. (1920)**
*On the Rate of Growth of the Population of the United States Since 1790 and Its Mathematical Representation.*
Proceedings of the National Academy of Sciences, 6(6), 275–288. [DOI](https://doi.org/10.1073/pnas.6.6.275)
→ Introduced the logistic model as a direct competitor to exponential growth — shows how quickly Malthus's limitations became apparent empirically.

**3. United Nations (2022)**
*World Population Prospects 2022: Summary of Results.*
[Link](https://www.un.org/development/desa/pd/sites/www.un.org.development.desa.pd/files/wpp2022_summary_of_results.pdf)
→ The authoritative source for the population data you will use. Pay attention to the methodology section — understanding where data comes from is part of doing good science.

---

### Journal Club — Modern Perspectives

**4. Reher, D. S. (2004)**
*The Demographic Transition Revisited as a Global Process.*
Population, Space and Place, 10(1), 19–41. [DOI](https://doi.org/10.1002/psp.313)
→ Directly relevant to the Demographic Transition analytical extension. Asks whether the Western demographic transition experience generalises globally.

**5. Cohen, J. E. (1995)**
*Population Growth and Earth's Human Carrying Capacity.*
Science, 269(5222), 341–346. [DOI](https://doi.org/10.1126/science.7618100)
→ Classic review putting the Malthusian vs. logistic debate in sharp relief. Worth reading before tackling the Comparative Model Analysis extension.

**6. Vollset, S. E. et al. (2020)**
*Fertility, Mortality, Migration, and Population Scenarios for 195 Countries and Territories from 2017 to 2100.*
The Lancet, 396(10258), 1285–1306. [DOI](https://doi.org/10.1016/S0140-6736(20)30677-2)
→ State-of-the-art projections predicting a world population peak around 2064 followed by decline — a stark departure from Malthusian growth. Good reading for the limitations section of your report.